# 07 - API and Deployment

Smoke-test the FastAPI `/predict` endpoint locally. Walk through Render deployment steps and health-check verification.

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path
from sklearn.model_selection import train_test_split

MODELS_PATH = Path('../outputs/models')
THRESHOLD   = 0.28   # optimal threshold from notebook 06


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\ProgramData\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\ProgramData\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\ProgramData\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\New User\AppData\Roaming\Pyth

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\ProgramData\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\ProgramData\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\ProgramData\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\New User\AppData\Roaming\Pyth

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



## 1. Verify model artifact

In [2]:
model = joblib.load(MODELS_PATH / 'best_xgb.pkl')

print(f'Model type   : {type(model).__name__}')
print(f'Features     : {model.n_features_in_}')
print(f'Threshold    : {THRESHOLD}')
print(f'Model file   : {(MODELS_PATH / "best_xgb.pkl").stat().st_size / 1024:.0f} KB')

Model type   : XGBClassifier
Features     : 30
Threshold    : 0.28
Model file   : 915 KB


## 2. Prediction function

A thin wrapper used by both the notebook smoke-test and the FastAPI app.

In [3]:
def predict_transaction(transaction: dict, threshold: float = THRESHOLD) -> dict:
    """Score a single transaction. Input keys must match training features."""
    df   = pd.DataFrame([transaction])
    prob = float(model.predict_proba(df)[:, 1][0])
    return {
        "fraud_probability": round(prob, 4),
        "is_fraud":          bool(prob >= threshold),
        "threshold_used":    threshold,
    }

print("predict_transaction() ready")

predict_transaction() ready


## 3. Smoke-test - single transactions

In [4]:
df_raw = pd.read_csv('../data/raw/creditcard.csv')
df_raw['log_amount']  = np.log1p(df_raw['Amount'])
df_raw['hour_of_day'] = (df_raw['Time'] // 3600) % 24
df_raw = df_raw.drop(columns=['Time', 'Amount'])

X = df_raw.drop(columns=['Class'])
y = df_raw['Class']

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Pick one known fraud and one known legit transaction
fraud_idx = y_test[y_test == 1].index[0]
legit_idx = y_test[y_test == 0].index[0]

fraud_result = predict_transaction(X_test.loc[fraud_idx].to_dict())
legit_result = predict_transaction(X_test.loc[legit_idx].to_dict())

print("Known FRAUD transaction:")
print(json.dumps(fraud_result, indent=2))
print()
print("Known LEGIT transaction:")
print(json.dumps(legit_result, indent=2))

Known FRAUD transaction:
{
  "fraud_probability": 0.9999,
  "is_fraud": true,
  "threshold_used": 0.28
}

Known LEGIT transaction:
{
  "fraud_probability": 0.0,
  "is_fraud": false,
  "threshold_used": 0.28
}


## 4. Batch scoring

In [5]:
sample = X_test.head(200)
results = [predict_transaction(row.to_dict()) for _, row in sample.iterrows()]

flagged    = sum(r['is_fraud'] for r in results)
true_fraud = int(y_test.loc[sample.index].sum())

print(f'Transactions scored : {len(results)}')
print(f'Flagged as fraud    : {flagged}')
print(f'Actual fraud        : {true_fraud}')
print(f'Score range         : {min(r["fraud_probability"] for r in results):.4f} - '
      f'{max(r["fraud_probability"] for r in results):.4f}')

Transactions scored : 200
Flagged as fraud    : 1
Actual fraud        : 0
Score range         : 0.0000 - 0.9950


## 5. FastAPI app

The code below shows the complete `app/main.py`. It is written to disk here so it is ready to deploy.

In [6]:
app_code = '''from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd
import joblib
from pathlib import Path

app   = FastAPI(title="Fraud Detection API", version="1.0")
MODEL = joblib.load(Path("outputs/models/best_xgb.pkl"))
THRESHOLD = 0.28


class Transaction(BaseModel):
    # PCA components from the credit card dataset
    V1: float; V2: float; V3: float; V4: float; V5: float
    V6: float; V7: float; V8: float; V9: float; V10: float
    V11: float; V12: float; V13: float; V14: float; V15: float
    V16: float; V17: float; V18: float; V19: float; V20: float
    V21: float; V22: float; V23: float; V24: float; V25: float
    V26: float; V27: float; V28: float
    log_amount: float
    hour_of_day: float


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(txn: Transaction):
    df   = pd.DataFrame([txn.model_dump()])
    prob = float(MODEL.predict_proba(df)[:, 1][0])
    return {
        "fraud_probability": round(prob, 4),
        "is_fraud":          bool(prob >= THRESHOLD),
        "threshold_used":    THRESHOLD,
    }
'''

app_path = Path('../app')
app_path.mkdir(exist_ok=True)
(app_path / '__init__.py').touch()
(app_path / 'main.py').write_text(app_code)
print("Written: app/main.py")
print()
print(app_code)

Written: app/main.py

from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd
import joblib
from pathlib import Path

app   = FastAPI(title="Fraud Detection API", version="1.0")
MODEL = joblib.load(Path("outputs/models/best_xgb.pkl"))
THRESHOLD = 0.28


class Transaction(BaseModel):
    # PCA components from the credit card dataset
    V1: float; V2: float; V3: float; V4: float; V5: float
    V6: float; V7: float; V8: float; V9: float; V10: float
    V11: float; V12: float; V13: float; V14: float; V15: float
    V16: float; V17: float; V18: float; V19: float; V20: float
    V21: float; V22: float; V23: float; V24: float; V25: float
    V26: float; V27: float; V28: float
    log_amount: float
    hour_of_day: float


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(txn: Transaction):
    df   = pd.DataFrame([txn.model_dump()])
    prob = float(MODEL.predict_proba(df)[:, 1][0])
    return {
        "fraud_probability

## 6. Render deployment

### Files needed in repo root
| File | Purpose |
|------|---------|
| `requirements.txt` | Python dependencies |
| `app/main.py` | FastAPI application |
| `outputs/models/best_xgb.pkl` | Trained model |

### Steps
1. Push repo to GitHub
2. Go to [render.com](https://render.com) → **New Web Service**
3. Connect the GitHub repo
4. Set build command: `pip install -r requirements.txt`
5. Set start command: `uvicorn app.main:app --host 0.0.0.0 --port $PORT`
6. Deploy

### Testing the live endpoint
```bash
curl -X POST https://your-app.onrender.com/predict \
  -H "Content-Type: application/json" \
  -d '{"V1": -1.36, "V2": -0.07, ..., "log_amount": 4.5, "hour_of_day": 14}'
```

## 7. Project summary

In [7]:
print('=== FRAUD DETECTION PIPELINE - COMPLETE ===')
print()
print('Notebooks:')
print('  01  EDA (class imbalance, V-feature distributions)')
print('  02  Baseline logistic regression (PR-AUC 0.7315)')
print('  03  Imbalance handling (SMOTE + class weights, PR-AUC 0.7949)')
print('  04  XGBoost + Optuna tuning (PR-AUC 0.8795, MLflow)')
print('  05  SHAP explainability (V4 top fraud signal, PR-AP)')
print('  06  Threshold optimisation (business cost matrix)')
print('  07  Model serving (FastAPI)')
print('       Cloud deployment (Render)')

=== FRAUD DETECTION PIPELINE - COMPLETE ===

Notebooks:
  01  EDA (class imbalance, V-feature distributions)
  02  Baseline logistic regression (PR-AUC 0.7315)
  03  Imbalance handling (SMOTE + class weights, PR-AUC 0.7949)
  04  XGBoost + Optuna tuning (PR-AUC 0.8795, MLflow)
  05  SHAP explainability (V4 top fraud signal, PR-AP)
  06  Threshold optimisation (business cost matrix)
  07  Model serving (FastAPI)
       Cloud deployment (Render)
